# 00 — Download Goodreads data (all genres)

Direct HTTP download from the UCSD mirror (the official method) — **no gdown / Drive links needed**. Files are served at `https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/` (per-genre files under `byGenre/`). Saves into `data/` with the names `02_preprocess` expects.

All-genre raw is ~15–30 GB (fine on a 477 GB SSD or Kaggle's session disk). For a faster first run, trim the `GENRES` list.

In [ ]:
import os, requests
from tqdm.auto import tqdm

In [ ]:
BASE = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/"
GENRES = ["children", "comics_graphic", "fantasy_paranormal", "history_biography",
          "mystery_thriller_crime", "poetry", "romance", "young_adult"]

def download(url, out):
    if os.path.exists(out):
        print("skip (exists):", out)
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0)) or None
        with open(out, "wb") as f, tqdm(total=total, unit="B", unit_scale=True,
                                        desc=os.path.basename(out)) as bar:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                bar.update(len(chunk))

In [ ]:
os.makedirs("data", exist_ok=True)
for g in GENRES:
    download(BASE + f"byGenre/goodreads_interactions_{g}.json.gz", f"data/interactions_{g}.json.gz")
download(BASE + "goodreads_books.json.gz", "data/books.json.gz")  # full graph = all genres
print(sorted(os.listdir("data")))

**Next:** run `02_preprocess_kaggle.ipynb` (streams `data/interactions_*.json.gz` → `artifacts/sample.parquet` + `catalog.parquet`).